# Marketing Goals — Combined (RP + LS) · **Colab** · RP `app_affiliate` → Affiliate

*Lee Jerusalmy*

Separate experiment. **Not** the current Combined freeze (v3 stays as-is).

Open in Colab: Runtime → Run all. Do not treat the export as production until Lee says so.

### What changed vs v3
- **RealPrize only:** `channel_type = app_affiliate` → Goals **Affiliate**. Remaining `marketing_population = APP` stays **App**.
- Those movers are `scope = non_app` and `bucket = acquired`.
- **LoneStar:** same mapping as v3 (`APP` and `cost_date >= 2026-08-05` → App). No `app_affiliate` carve-out.
- Everything else is a copy of v3: capped winsor_esc, LS App `native_early_rp_tail`, App organic off, `AS_OF_DATE` pinned **2026-08-19**.

### Locked main columns
`brand`, `population`, `goal_horizon`, `day`, `raw_goal_ratio`, `organic_share`, `adjusted_goal_ratio`

### How to run
1. **Runtime → Run all** (or top → bottom).
2. First cell: click allow for Google auth.
3. Optional: mount Drive to write under `lee_project/marketing_goals/runs/`.
4. Export tag: `rp_app_affiliate_to_aff` (does not overwrite the v3 freeze folder).

Do not edit `reference/` or `notebooks/versions/v3_2026-08_winsor_esc_ls_app/`. Generic `notebooks/` Combined stays baseline.


In [1]:
# ── Auth (Google Colab) ──────────────────────────────────────────
from google.colab import auth
auth.authenticate_user()
IN_COLAB = True
print('Authenticated (Colab)')


Authenticated (Colab)


In [2]:
project_id = 'oceanic-citadel-454608-d2'
from google.cloud import bigquery
client = bigquery.Client(project=project_id)
print('BigQuery client ready for', project_id)


BigQuery client ready for oceanic-citadel-454608-d2


In [3]:
# ── Optional: mount Google Drive (to write under marketing_goals/runs/) ──
# Skip this cell if you only want browser downloads.

MOUNT_DRIVE = True  # set False to skip

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted. Expected runs path:')
    print('  /content/drive/MyDrive/lee_project/marketing_goals/runs')
    print('  (or My Drive with a space — export cell checks both)')
else:
    print('Drive mount skipped — export will browser-download CSVs only.')


Mounted at /content/drive
Drive mounted. Expected runs path:
  /content/drive/MyDrive/lee_project/marketing_goals/runs
  (or My Drive with a space — export cell checks both)


In [4]:
import pandas as pd
import numpy as np
from pandas_gbq import read_gbq

# ── Date anchor (shared for both brands) ──
# ADDED: pinned to 2026-08-11 so cohort_start/end per patch match runs/2026-08-03_rp_ls
# for before/after comparison of the adaptive-winsor test. REVERT to rolling
# `pd.Timestamp.now().normalize() - pd.Timedelta(days=2)` after this test round
# and once the fix is adopted for production runs.
AS_OF_DATE = pd.Timestamp('2026-08-19').normalize()  # pinned to match v3 freeze export
# THIS EXPERIMENT (RP only): channel_type = app_affiliate → Affiliate.
# Remaining APP stays App. LoneStar mapping unchanged. See load_brand_tables.
# LS App population floor (this experiment). User life clock stays cost_date;
# affid=1 before this date is not App (falls through to Affiliate).
LS_APP_START_DATE = pd.Timestamp('2026-08-05').normalize()

# ── Shared patch / horizon calendar ──
PATCHES = (
    (1, 7), (7, 14), (14, 30), (30, 60), (60, 90),
    (90, 120), (120, 150), (150, 180), (180, 270), (270, 365),
)
GOAL_HORIZONS = [7, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]
CHECKPOINTS   = [7, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]
LOOKBACK_COHORTS = 35
ORGANIC_LABEL = 'Organic'

# ── Which brands to run ──
RUN_BRANDS = ['realprize', 'lonestar']

# ── Per-brand knobs (from config/*.yaml / Combined predecessors) ──
BRAND_CONFIGS = {
    'realprize': {
        'brand': 'realprize',
        'cost_table': 'analytics.realprize_cost_per_user',
        'deposits_table': 'realprize.casino_astropay_dmn',
        # TikTok excluded entirely
        'exclude_affids': [4313],
        'populations': ['Web', 'App', 'Affiliate'],
        'trim_config': {
            'App':       {'method': 'winsor', 'pct': 0},
            'Web':       {'method': 'winsor', 'pct': 0.01},
            'Affiliate': {'method': 'winsor', 'pct': 0.01},
            'Blended':   {'method': 'winsor', 'pct': 0},
        },
        'organic_trim_method': 'winsor',
        'organic_trim_pct': 0,
        # App attribution change ~2025-08-12 — pin share above this horizon
        'organic_share_cap_horizon': 120,
        'cv_threshold': 0.15,
        'cv_good_enough': 0.10,
        'max_remove_fraction': 0.15,
        # RP Combined did not use a min-cohort-dates gate (default 1)
        'min_cohort_dates': 1,
        # Tail fill only in LS predecessor
        'extrapolate_tail': False,
        'extrapolation_tail_days': 30,
        'has_scope_bucket': True,  # app / non_app + app_organic
        'users_sql': None,  # filled below via builder
    },
    'lonestar': {
        'brand': 'lonestar',
        'cost_table': 'analytics.lonestar_cost_per_user',
        'deposits_table': 'lonestar.casino_astropay_dmn',
        # TikTok + TikTok Canada
        'exclude_affids': [4866, 7127],
        'populations': ['Web', 'Affiliate'],  # App is an add-on in the RUN cell; not in Blended
        'trim_config': {
            'Web':       {'method': 'winsor', 'pct': 0},
            'App':       {'method': 'winsor', 'pct': 0},  # locked 0%; RUN skips escalation
            'Affiliate': {'method': 'winsor', 'pct': 0.01},
            'Blended':   {'method': 'winsor', 'pct': 0},
        },
        'organic_trim_method': 'winsor',
        'organic_trim_pct': 0,
        # No organic share cap on LS
        'organic_share_cap_horizon': None,
        'cv_threshold': 0.15,
        'cv_good_enough': 0.10,
        'max_remove_fraction': 0.15,
        'min_cohort_dates': 20,  # Web / Affiliate / Blended. App override = 1 in RUN cell.
        'extrapolate_tail': True,
        'extrapolation_tail_days': 30,
        'has_scope_bucket': True,  # app / non_app + app_organic (same idea as RP)
        'users_sql': None,
    },
}

# ── Monitoring (print sub-outputs after each pipeline step) ──
# True  = show intermediate tables in Colab after each part (recommended first runs)
# False = quieter (still prints step banners + row counts)
MONITOR_STEPS = True
MONITOR_PREVIEW_DAYS = [1, 7, 14, 30, 60, 90, 120, 180, 270, 365]
MONITOR_PREVIEW_HORIZONS = [7, 30, 120, 365]  # goals sample only

print('ENV: Google Colab')

# ── Seed Combined globals (overwritten per brand in apply_brand_globals) ──
# Helper cells use these names as defaults; they MUST exist before those cells run.
POPULATIONS = list(BRAND_CONFIGS['realprize']['populations'])  # first brand's list as placeholder
TRIM_CONFIG = dict(BRAND_CONFIGS['realprize']['trim_config'])
ORGANIC_TRIM_METHOD = BRAND_CONFIGS['realprize']['organic_trim_method']
ORGANIC_TRIM_PCT = BRAND_CONFIGS['realprize']['organic_trim_pct']
ORGANIC_SHARE_CAP_HORIZON = BRAND_CONFIGS['realprize']['organic_share_cap_horizon']
CV_THRESHOLD = BRAND_CONFIGS['realprize']['cv_threshold']
CV_GOOD_ENOUGH = BRAND_CONFIGS['realprize']['cv_good_enough']
MAX_REMOVE_FRACTION = BRAND_CONFIGS['realprize']['max_remove_fraction']
MIN_COHORT_DATES = BRAND_CONFIGS['realprize']['min_cohort_dates']
EXTRAPOLATE_TAIL = BRAND_CONFIGS['realprize']['extrapolate_tail']
EXTRAPOLATION_TAIL_DAYS = BRAND_CONFIGS['realprize']['extrapolation_tail_days']
ACTIVE_BRAND = None

print(f'Config loaded. as_of_date = {AS_OF_DATE.date()}')
print(f'Brands to run: {RUN_BRANDS}')
print(f'Monitor steps = {MONITOR_STEPS}')
print('Mode: PERSISTENT TRIM (excluded users carry forward)')
print('Main goals columns: brand, population, goal_horizon, day, raw_goal_ratio, organic_share, adjusted_goal_ratio')


ENV: Google Colab
Config loaded. as_of_date = 2026-08-19
Brands to run: ['realprize', 'lonestar']
Monitor steps = True
Mode: PERSISTENT TRIM (excluded users carry forward)
Main goals columns: brand, population, goal_horizon, day, raw_goal_ratio, organic_share, adjusted_goal_ratio


In [5]:
# ══════════════════════════════════════════════════════════════
# APPLY BRAND CONFIG → module-level globals used by helpers
# (same variable names as Combined predecessors)
# ══════════════════════════════════════════════════════════════

def apply_brand_globals(cfg: dict) -> None:
    """Set Combined-style globals for one brand before running the pipeline."""
    global POPULATIONS, TRIM_CONFIG
    global ORGANIC_TRIM_METHOD, ORGANIC_TRIM_PCT, ORGANIC_SHARE_CAP_HORIZON
    global CV_THRESHOLD, CV_GOOD_ENOUGH, MAX_REMOVE_FRACTION
    global MIN_COHORT_DATES, EXTRAPOLATE_TAIL, EXTRAPOLATION_TAIL_DAYS
    global ACTIVE_BRAND

    ACTIVE_BRAND = cfg['brand']
    POPULATIONS = list(cfg['populations'])
    TRIM_CONFIG = dict(cfg['trim_config'])
    ORGANIC_TRIM_METHOD = cfg['organic_trim_method']
    ORGANIC_TRIM_PCT = cfg['organic_trim_pct']
    ORGANIC_SHARE_CAP_HORIZON = cfg['organic_share_cap_horizon']
    CV_THRESHOLD = cfg['cv_threshold']
    CV_GOOD_ENOUGH = cfg['cv_good_enough']
    MAX_REMOVE_FRACTION = cfg['max_remove_fraction']
    MIN_COHORT_DATES = cfg['min_cohort_dates']
    EXTRAPOLATE_TAIL = cfg['extrapolate_tail']
    EXTRAPOLATION_TAIL_DAYS = cfg['extrapolation_tail_days']

    print(f"\n{'=' * 60}")
    print(f"BRAND: {ACTIVE_BRAND}")
    print(f"  populations = {POPULATIONS}")
    print(f"  CV threshold = {CV_THRESHOLD}  good_enough = {CV_GOOD_ENOUGH}")
    print(f"  min_cohort_dates = {MIN_COHORT_DATES}")
    print(f"  extrapolate_tail = {EXTRAPOLATE_TAIL}")
    print(f"  organic_share_cap_horizon = {ORGANIC_SHARE_CAP_HORIZON}")
    print(f"{'=' * 60}")


def load_brand_tables(cfg: dict, as_of_date=AS_OF_DATE):
    """Pull cost-per-user population assignment + deposit revenue for one brand."""
    sql_floor = (
        as_of_date - pd.Timedelta(days=max(GOAL_HORIZONS) + LOOKBACK_COHORTS + 5)
    ).date()
    brand = cfg['brand']
    cost_table = cfg['cost_table']
    dep_table = cfg['deposits_table']
    excl = cfg['exclude_affids']
    excl_sql = ', '.join(str(a) for a in excl)

    print(f'[{brand}] SQL floor date: {sql_floor}  (AS_OF_DATE = {as_of_date.date()})')
    if brand == 'lonestar':
        print(f'[{brand}] LS App start_date: {LS_APP_START_DATE.date()}  '
              f'(affid=1 before this date is Affiliate, not App)')

    if brand == 'realprize':
        # Population from cost_per_user.marketing_population (channel map).
        # Exceptions: SEO / Shared Link / affid 2290 → Organic. TikTok still excluded.
        # THIS EXPERIMENT (RP only): channel_type = app_affiliate → Affiliate.
        # Remaining APP stays App. LoneStar mapping is unchanged.
        users_sql = f"""
        WITH base AS (
          SELECT
            id,
            affid,
            channel_type,
            marketing_population,
            DATE(MIN(cost_date)) AS cost_date
          FROM `{cost_table}`
          WHERE cost_date >= DATE('{sql_floor}')
            AND affid NOT IN ({excl_sql})
            AND id > 0
          GROUP BY id, affid, channel_type, marketing_population
        )
        SELECT
          id,
          CASE
            WHEN affid = 2290 THEN 'Organic'
            WHEN marketing_population IN ('SEO', 'Shared Link', 'Organic') THEN 'Organic'
            WHEN marketing_population = 'WEB' THEN 'Web'
            WHEN channel_type = 'app_affiliate' THEN 'Affiliate'
            WHEN marketing_population = 'APP' THEN 'App'
            WHEN marketing_population IN ('Google PPC', 'Bing PPC', 'PPC') THEN 'PPC'
            ELSE 'Affiliate'
          END AS population,
          CASE
            WHEN channel_type = 'app_affiliate' THEN 'non_app'
            WHEN marketing_population = 'APP' THEN 'app'
            ELSE 'non_app'
          END AS scope,
          CASE
            WHEN channel_type = 'app_affiliate' THEN 'acquired'
            WHEN marketing_population = 'APP' AND channel_type = 'app_organic' THEN 'organic'
            WHEN marketing_population = 'APP' THEN 'acquired'
            WHEN affid = 2290 THEN 'organic'
            WHEN marketing_population IN ('SEO', 'Shared Link', 'Organic') THEN 'organic'
            ELSE 'acquired'
          END AS bucket,
          cost_date
        FROM base
        """
    elif brand == 'lonestar':
        # Same scope/bucket idea as RP so App gets its own organic share.
        # Population from marketing_population; LS App still floored at LS_APP_START_DATE.
        app_floor = LS_APP_START_DATE.date()
        users_sql = f"""
        WITH base AS (
          SELECT
            id,
            affid,
            channel_type,
            marketing_population,
            DATE(MIN(cost_date)) AS cost_date
          FROM `{cost_table}`
          WHERE cost_date >= DATE('{sql_floor}')
            AND affid NOT IN ({excl_sql})
            AND id > 0
          GROUP BY id, affid, channel_type, marketing_population
        )
        SELECT
          id,
          CASE
            WHEN marketing_population IN ('SEO', 'Shared Link', 'Organic') THEN 'Organic'
            WHEN marketing_population = 'WEB' THEN 'Web'
            WHEN marketing_population = 'APP' AND cost_date >= DATE('{app_floor}') THEN 'App'
            WHEN marketing_population IN ('Google PPC', 'Bing PPC', 'PPC') THEN 'PPC'
            ELSE 'Affiliate'
          END AS population,
          CASE
            WHEN marketing_population = 'APP' AND cost_date >= DATE('{app_floor}') THEN 'app'
            ELSE 'non_app'
          END AS scope,
          CASE
            WHEN marketing_population = 'APP' AND cost_date >= DATE('{app_floor}')
                 AND channel_type = 'app_organic' THEN 'organic'
            WHEN marketing_population = 'APP' AND cost_date >= DATE('{app_floor}') THEN 'acquired'
            WHEN marketing_population IN ('SEO', 'Shared Link', 'Organic') THEN 'organic'
            ELSE 'acquired'
          END AS bucket,
          cost_date
        FROM base
        """
    else:
        raise ValueError(f'Unknown brand: {brand}')

    revenue_sql = f"""
    SELECT
      playerId AS playerid,
      DATE(date) AS date,
      SUM(amount) / 100.0 AS amount
    FROM `{dep_table}`
    WHERE Status = 'APPROVED'
      AND date >= DATE('{sql_floor}')
    GROUP BY 1, 2
    """

    users_df = read_gbq(users_sql, project_id=project_id, use_bqstorage_api=True)
    revenue_df = read_gbq(revenue_sql, project_id=project_id, use_bqstorage_api=True)

    print(f'[{brand}] users_df:   {len(users_df):,} rows')
    print(f'[{brand}] revenue_df: {len(revenue_df):,} rows')
    print(users_df['population'].value_counts().to_string())
    if brand == 'realprize':
        print('[realprize] experiment: channel_type=app_affiliate → Affiliate; '
              'other APP stays App. LoneStar unchanged.')
        print(users_df.groupby(['population', 'scope', 'bucket']).size().to_string())
    return users_df, revenue_df

print('Brand loader + apply_brand_globals ready.')


Brand loader + apply_brand_globals ready.


In [6]:
# ══════════════════════════════════════════════════════════════
# HELPERS — math & base table construction
# ══════════════════════════════════════════════════════════════

def weighted_mean_std_cv(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    m = np.isfinite(x) & np.isfinite(w) & (w > 0)
    x, w = x[m], w[m]
    if x.size == 0:
        return np.nan, np.nan, np.nan
    mu  = np.average(x, weights=w)
    var = np.average((x - mu) ** 2, weights=w)
    sd  = np.sqrt(var)
    cv  = sd / mu if mu != 0 else np.nan
    return mu, sd, cv


def build_user_revenue_cums(users_df, revenue_df, *, max_day=365):
    """
    Precompute cumulative per-user revenue indexed by (population, cost_date, user, dsi).
    Returns:
      u          — one row per (population, user) with their earliest cost_date
      daily_user — cumulative revenue at each (population, cost_date, user, dsi)
    """
    u = users_df[['id', 'population', 'cost_date']].copy()
    u['population'] = u['population'].astype(str).str.strip()
    u['cost_date']  = pd.to_datetime(u['cost_date'], errors='coerce').dt.date
    u = u.loc[pd.notna(u['cost_date'])].copy()
    u = u.groupby(['population', 'id'], as_index=False)['cost_date'].min()
    u = u.rename(columns={'id': '__uid__'})

    r = revenue_df[['playerid', 'date', 'amount']].copy()
    r['date'] = pd.to_datetime(r['date'], errors='coerce').dt.date
    r = r.loc[pd.notna(r['date'])].copy()

    rr = r.merge(u, left_on='playerid', right_on='__uid__', how='inner')
    rr['dsi'] = (pd.to_datetime(rr['date']) - pd.to_datetime(rr['cost_date'])).dt.days
    rr = rr.loc[(rr['dsi'] >= 0) & (rr['dsi'] <= (max_day - 1))].copy()

    daily_user = (
        rr.groupby(['population', 'cost_date', '__uid__', 'dsi'], observed=True)['amount']
          .sum().reset_index()
          .sort_values(['population', 'cost_date', '__uid__', 'dsi'])
    )
    daily_user['cum_amount'] = (
        daily_user.groupby(['population', 'cost_date', '__uid__'], observed=True)['amount']
                  .cumsum()
    )
    return u, daily_user


print('Math + base table helpers defined.')


Math + base table helpers defined.


In [7]:
# ══════════════════════════════════════════════════════════════
# HELPERS — trimming & cohort revenue summation
# ══════════════════════════════════════════════════════════════

def compute_winsor_caps(daily_user_cums, cohort_users, e, top_pct=0.01):
    """Cap per-user cumulative revenue at the (1-top_pct) quantile within each cohort date."""
    if top_pct <= 0:
        caps = cohort_users[['population', 'cost_date', '__uid__']].copy()
        caps['cap_e'] = np.inf
        return caps
    du = daily_user_cums.loc[daily_user_cums['dsi'] <= (e - 1)].copy()
    if du.empty:
        caps = cohort_users[['population', 'cost_date', '__uid__']].copy()
        caps['cap_e'] = np.inf
        return caps
    per_user = (
        du.groupby(['population', 'cost_date', '__uid__'], observed=True)['cum_amount']
          .max().reset_index(name='cum_e')
    )
    per_user = cohort_users.merge(per_user, on=['population', 'cost_date', '__uid__'], how='left')
    per_user['cum_e'] = per_user['cum_e'].fillna(0.0)
    per_user['cap_e'] = (
        per_user.groupby(['population', 'cost_date'], observed=True)['cum_e']
                .transform(lambda s: s[s > 0].quantile(1.0 - top_pct) if (s > 0).any() else np.inf)
    )
    return per_user[['population', 'cost_date', '__uid__', 'cap_e']]


def apply_cohort_trim(daily_user_cums, cohort_users, e, trim_pct=0.10):
    """Remove the top trim_pct of depositors (by cumulative revenue) from each cohort date."""
    du = daily_user_cums.loc[daily_user_cums['dsi'] <= (e - 1)].copy()
    if du.empty:
        return cohort_users.copy()
    per_user = (
        du.groupby(['population', 'cost_date', '__uid__'], observed=True)['cum_amount']
          .max().reset_index(name='cum_e')
    )
    per_user = cohort_users.merge(per_user, on=['population', 'cost_date', '__uid__'], how='left')
    per_user['cum_e'] = per_user['cum_e'].fillna(0.0)
    depositors = per_user.loc[per_user['cum_e'] > 0].copy()
    if depositors.empty:
        return cohort_users.copy()
    thresholds = (
        depositors.groupby(['population', 'cost_date'], observed=True)['cum_e']
                  .quantile(1.0 - trim_pct).reset_index(name='threshold')
    )
    per_user = per_user.merge(thresholds, on=['population', 'cost_date'], how='left')
    per_user['threshold'] = per_user['threshold'].fillna(np.inf)
    keep = per_user.loc[
        (per_user['cum_e'] == 0) | (per_user['cum_e'] <= per_user['threshold'])
    ][['population', 'cost_date', '__uid__']]
    return keep.copy()


def get_trimmed_cohort_and_caps(population, cohort_users, daily_user_cums, e, pct_override=None):
    # CHANGED: optional pct_override for per-patch winsor escalation; now returns method, pct_used
    cfg = TRIM_CONFIG.get(population, {'method': 'cohort_trim', 'pct': 0.10})
    method = cfg['method']
    caps, trimmed = None, cohort_users
    if method == 'winsor':
        # ADDED: when pct_override is set, use it instead of config floor (winsor only)
        pct_used = cfg['pct'] if pct_override is None else pct_override
        caps = compute_winsor_caps(daily_user_cums, cohort_users, e, top_pct=pct_used)
    elif method == 'cohort_trim':
        # unchanged: no escalation for cohort_trim
        pct_used = cfg['pct']
        trimmed = apply_cohort_trim(daily_user_cums, cohort_users, e, trim_pct=pct_used)
    else:
        pct_used = cfg.get('pct', 0)
    return trimmed, caps, method, pct_used


def sum_cum_at_idx(daily_user_cums, *, cohort_users, idx, caps=None):
    """Sum cumulative revenue across all users in cohort_users at day index idx."""
    grp_cols = ['population', 'cost_date']
    if idx < 0:
        out = cohort_users.groupby(grp_cols, observed=True)['__uid__'].nunique().reset_index()
        out['sum_cum'] = 0.0
        return out[grp_cols + ['sum_cum']]
    du = daily_user_cums.loc[daily_user_cums['dsi'] <= idx].copy()
    if du.empty:
        out = cohort_users.groupby(grp_cols, observed=True)['__uid__'].nunique().reset_index()
        out['sum_cum'] = 0.0
        return out[grp_cols + ['sum_cum']]
    per_user = (
        du.groupby(grp_cols + ['__uid__'], observed=True)['cum_amount']
          .max().reset_index(name='cum')
    )
    per_user = cohort_users.merge(per_user, on=grp_cols + ['__uid__'], how='left')
    per_user['cum'] = per_user['cum'].fillna(0.0)
    if caps is not None:
        per_user = per_user.merge(caps, on=grp_cols + ['__uid__'], how='left')
        per_user['cap_e'] = per_user['cap_e'].fillna(np.inf)
        per_user['cum']   = np.minimum(per_user['cum'], per_user['cap_e'])
    sums = per_user.groupby(grp_cols, observed=True)['cum'].sum().reset_index(name='sum_cum')
    return sums


print('Trim + summation helpers defined.')


Trim + summation helpers defined.


In [8]:
# ══════════════════════════════════════════════════════════════
# ADAPTIVE CV ANALYSIS — persistent trim
# ══════════════════════════════════════════════════════════════

# ADDED: winsor escalation ladder (per patch). Start at TRIM_CONFIG floor; climb
# only until cv_after <= CV_THRESHOLD. Never go below floor; never overshoot past first clear.
WINSOR_ESCALATION_STEPS = [0.0, 0.005, 0.01, 0.02, 0.03, 0.05]

# ADDED: second stop — do not escalate into a pct that caps away too much revenue.
# If a candidate would cut more than this fraction of total_rev_before_trim, reject it
# and keep the previous (lower) pct's result (even if still flagged).
MAX_REVENUE_CUT_FRACTION = 0.15


def patch_cv_adaptive(
    u_base, daily_user_cums, *,
    population, s, e, as_of_date,
    excluded_uids=None,
    lookback_cohorts=None,
    cv_threshold=None,
    cv_good_enough=None,
    max_remove_fraction=None,
    debug=True,
):
    # Resolve defaults at call time (after brand config applied)
    if lookback_cohorts is None:
        lookback_cohorts = LOOKBACK_COHORTS
    if cv_threshold is None:
        cv_threshold = CV_THRESHOLD
    if cv_good_enough is None:
        cv_good_enough = CV_GOOD_ENOUGH
    if max_remove_fraction is None:
        max_remove_fraction = MAX_REMOVE_FRACTION
    as_of_date   = pd.to_datetime(as_of_date).normalize()
    cohort_end   = (as_of_date - pd.Timedelta(days=e)).date()
    cohort_start = (as_of_date - pd.Timedelta(days=e + (lookback_cohorts - 1))).date()

    cohort_users = u_base.loc[
        (u_base['population'] == population) &
        (u_base['cost_date'] >= cohort_start) &
        (u_base['cost_date'] <= cohort_end)
    ][['population', 'cost_date', '__uid__']].copy()

    if cohort_users.empty:
        return pd.DataFrame(), {}, [], False, set()

    all_cohort_users = cohort_users.copy()
    n_users_in_cohort = int(cohort_users['__uid__'].nunique())

    if excluded_uids:
        cohort_users = cohort_users.loc[
            ~cohort_users['__uid__'].isin(excluded_uids)
        ].copy()

    n_users_after_prior = int(cohort_users['__uid__'].nunique())
    n_users_excluded_prior = n_users_in_cohort - n_users_after_prior

    if cohort_users.empty:
        return pd.DataFrame(), {}, [], False, set()

    # ADDED: winsor escalation — candidate pcts from config floor upward
    cfg = TRIM_CONFIG.get(population, {})
    floor_pct = cfg.get('pct', 0)
    method_cfg = cfg.get('method', 'none')
    if method_cfg != 'winsor':
        candidates = [floor_pct]
    else:
        candidates = sorted(set([floor_pct] + [p for p in WINSOR_ESCALATION_STEPS if p >= floor_pct]))

    sum_e_all = sum_cum_at_idx(
        daily_user_cums, cohort_users=all_cohort_users, idx=e - 1, caps=None
    ).rename(columns={'sum_cum': 'sum_cum_e_all'})
    total_rev_before_trim = float(sum_e_all['sum_cum_e_all'].sum())

    # Final state = last acceptable candidate (within revenue-cut limit), or first clear.
    patch = pd.DataFrame()
    remaining = pd.DataFrame()
    removed = []
    newly_excluded = set()
    n_users_pre_trim = n_users_after_prior
    n_users_post_trim = n_users_after_prior
    cv_before = np.nan
    cv_after = np.nan
    mean_a = np.nan
    flagged = False
    pct_used = floor_pct
    trim_method = method_cfg
    n_escalation_steps_tried = 0
    total_rev_after_trim = 0.0
    revenue_cut_fraction = 0.0
    capped_by_revenue_limit = False
    has_acceptable = False  # True once we have a result within the revenue-cut limit

    for pct_try in candidates:
        n_escalation_steps_tried += 1

        # CHANGED: unpack method + pct_used; pass pct_override for this attempt
        trimmed_users, caps, trim_method_try, pct_used_try = get_trimmed_cohort_and_caps(
            population, cohort_users, daily_user_cums, e, pct_override=pct_try
        )
        n_users_pre_trim_try  = n_users_after_prior
        n_users_post_trim_try = int(trimmed_users['__uid__'].nunique())

        newly_excluded_try = (
            set(cohort_users['__uid__'].unique()) - set(trimmed_users['__uid__'].unique())
        )

        denom_w = (
            trimmed_users.groupby(['population', 'cost_date'], observed=True)['__uid__']
                         .nunique().reset_index(name='N_users')
        )
        sum_s = sum_cum_at_idx(
            daily_user_cums, cohort_users=trimmed_users, idx=s - 1, caps=caps
        ).rename(columns={'sum_cum': 'sum_cum_s'})
        sum_e = sum_cum_at_idx(
            daily_user_cums, cohort_users=trimmed_users, idx=e - 1, caps=caps
        ).rename(columns={'sum_cum': 'sum_cum_e'})

        patch_try = (
            denom_w
            .merge(sum_s, on=['population', 'cost_date'])
            .merge(sum_e, on=['population', 'cost_date'])
        )
        patch_try['ARPU_s']       = patch_try['sum_cum_s'] / patch_try['N_users']
        patch_try['ARPU_e']       = patch_try['sum_cum_e'] / patch_try['N_users']
        patch_try['growth_ratio'] = np.where(
            patch_try['ARPU_s'] > 0, patch_try['ARPU_e'] / patch_try['ARPU_s'], np.nan
        )

        _, _, cv_before_try = weighted_mean_std_cv(
            patch_try['growth_ratio'].values, patch_try['sum_cum_s'].values
        )

        mu_unw = np.nanmean(patch_try['growth_ratio'].values)
        patch_try['abs_dev'] = (patch_try['growth_ratio'] - mu_unw).abs()
        sorted_dates  = patch_try.sort_values('abs_dev', ascending=False)['cost_date'].tolist()
        max_removable = max(1, int(np.floor(len(patch_try) * max_remove_fraction)))

        removed_try   = []
        remaining_try = patch_try.copy()

        for candidate in sorted_dates:
            _, _, cv_now = weighted_mean_std_cv(
                remaining_try['growth_ratio'].values, remaining_try['sum_cum_s'].values
            )
            if np.isnan(cv_now) or cv_now <= cv_good_enough:
                break
            if len(removed_try) >= max_removable:
                break
            removed_try.append(candidate)
            remaining_try = remaining_try.loc[~remaining_try['cost_date'].isin(removed_try)]

        mean_a_try, _, cv_after_try = weighted_mean_std_cv(
            remaining_try['growth_ratio'].values, remaining_try['sum_cum_s'].values
        )
        total_rev_after_trim_try = float(patch_try['sum_cum_e'].sum())
        # ADDED: revenue-cut check at EVERY candidate (independent of CV)
        if total_rev_before_trim > 0:
            revenue_cut_try = 1.0 - (total_rev_after_trim_try / total_rev_before_trim)
        else:
            revenue_cut_try = 0.0

        # STOP 1: this pct over-cuts revenue — reject it, keep previous acceptable result
        if revenue_cut_try > MAX_REVENUE_CUT_FRACTION:
            capped_by_revenue_limit = True
            if not has_acceptable:
                # Floor itself exceeds the cut limit: keep floor (no lower pct available)
                patch = patch_try
                remaining = remaining_try
                removed = removed_try
                newly_excluded = newly_excluded_try
                n_users_pre_trim = n_users_pre_trim_try
                n_users_post_trim = n_users_post_trim_try
                cv_before = cv_before_try
                cv_after = cv_after_try
                mean_a = mean_a_try
                total_rev_after_trim = total_rev_after_trim_try
                revenue_cut_fraction = revenue_cut_try
                pct_used = pct_used_try
                trim_method = trim_method_try
                flagged = (not np.isnan(cv_after)) and (cv_after > cv_threshold)
                has_acceptable = True
            # else: leave prior acceptable state unchanged
            break

        # Accept this candidate as current best within the revenue-cut limit
        patch = patch_try
        remaining = remaining_try
        removed = removed_try
        newly_excluded = newly_excluded_try
        n_users_pre_trim = n_users_pre_trim_try
        n_users_post_trim = n_users_post_trim_try
        cv_before = cv_before_try
        cv_after = cv_after_try
        mean_a = mean_a_try
        total_rev_after_trim = total_rev_after_trim_try
        revenue_cut_fraction = revenue_cut_try
        pct_used = pct_used_try
        trim_method = trim_method_try
        flagged = (not np.isnan(cv_after)) and (cv_after > cv_threshold)
        has_acceptable = True

        # STOP 2: CV cleared — first pct that works without over-cutting
        if (not np.isnan(cv_after)) and (cv_after <= cv_threshold):
            break
        # STOP 3: else if more candidates remain, loop continues; if last, keep this result

    escalated = bool(pct_used > floor_pct)

    if debug:
        flag_tag = f'  >>> FLAGGED (cv={cv_after:.4f} > {cv_threshold})' if flagged else ''
        esc_tag = f'  winsor={pct_used:.3f}' + (f' (escalated from {floor_pct:.3f})' if escalated else '')
        cut_tag = f'  rev_cut={revenue_cut_fraction:.1%}'
        if capped_by_revenue_limit:
            cut_tag += f'  >>> CAPPED BY REVENUE LIMIT ({MAX_REVENUE_CUT_FRACTION:.0%})'
        print(
            f'  [{population}] {s}->{e}  '
            f'cv {cv_before:.4f}->{cv_after:.4f}  '
            f'removed={len(removed)}/{len(patch)}  '
            f'excl_prior={n_users_excluded_prior:,}  '
            f'pre/post={n_users_pre_trim:,}/{n_users_post_trim:,}  '
            f'newly_excl={len(newly_excluded):,}{esc_tag}{cut_tag}{flag_tag}'
        )

    stats = dict(
        population              = population,
        patch                   = f'{s}->{e}',
        cohort_start            = str(cohort_start),
        cohort_end              = str(cohort_end),
        n_cohort_dates_total    = int(len(patch)),
        n_cohort_dates_kept     = int(len(remaining)),
        n_users_excluded_prior  = n_users_excluded_prior,
        n_users_pre_trim        = n_users_pre_trim,
        n_users_post_trim       = n_users_post_trim,
        n_users_dropped_by_trim = n_users_pre_trim - n_users_post_trim,
        total_rev_before_trim   = total_rev_before_trim,
        total_rev_after_trim    = total_rev_after_trim,
        cv_before               = float(cv_before) if not np.isnan(cv_before) else None,
        cv_after                = float(cv_after)  if not np.isnan(cv_after)  else None,
        mean_after              = float(mean_a)    if not np.isnan(mean_a)    else None,
        flagged                 = bool(flagged),
        removed_dates           = removed,
        trim_method             = trim_method,
        trim_pct                = pct_used,  # CHANGED: actual pct used (may be escalated)
        # ADDED: winsor escalation diagnostics
        floor_pct               = floor_pct,
        pct_used                = pct_used,
        escalated               = escalated,
        n_escalation_steps_tried = n_escalation_steps_tried,
        revenue_cut_fraction    = float(revenue_cut_fraction),
        capped_by_revenue_limit = bool(capped_by_revenue_limit),
    )
    return patch, stats, removed, flagged, newly_excluded


print('Adaptive CV function defined (persistent trim + winsor escalation + rev-cut cap).')


Adaptive CV function defined (persistent trim + winsor escalation + rev-cut cap).


In [9]:
# ══════════════════════════════════════════════════════════════
# PERSISTENT TRIM CURVE BUILDER (two-pass)
# ══════════════════════════════════════════════════════════════

def build_curve(
    u_base, daily_user_cums, *,
    population, as_of_date, debug=True,
):
    """Build ARPU curve with persistent trim — excluded users carry forward.
    Two-pass: first pass accumulates all exclusions unconditionally,
    second pass builds step ratios using frozen snapshots."""
    as_of_date = pd.to_datetime(as_of_date).normalize()

    cv_rows      = []
    effective    = []
    excluded_uids = set()

    # ── First pass: determine valid patches & accumulate exclusions ──
    for (s, e) in PATCHES:
        patch, stats, removed, flagged, newly_excluded = patch_cv_adaptive(
            u_base, daily_user_cums,
            population=population, s=s, e=e,
            as_of_date=as_of_date,
            excluded_uids=excluded_uids,
            debug=debug,
        )
        excluded_uids |= newly_excluded

        if not stats:
            continue

        min_cohort_dates = globals().get('MIN_COHORT_DATES', 1)
        if stats.get('n_cohort_dates_total', 0) < min_cohort_dates:
            if debug:
                print(f'  [{population}] {s}->{e}  insufficient data '
                      f'({stats["n_cohort_dates_total"]} < {min_cohort_dates}) — skipping')
            continue

        cv_rows.append(stats)
        effective.append({
            's': s, 'e': e,
            'removed_dates': removed,
            'excluded_snapshot': frozenset(excluded_uids),
        })

    if not effective:
        return pd.DataFrame(cv_rows), pd.DataFrame()

    if debug:
        print(f'  Total unique users excluded across all patches: {len(excluded_uids):,}')

    # ── Second pass: build step ratios using frozen snapshots ──
    step_rows = []
    for ep in effective:
        s, e      = ep['s'], ep['e']
        bad_dates = set(ep['removed_dates'])
        excl      = ep['excluded_snapshot']
        start_k   = 2 if s == 1 else (s + 1)

        cohort_end   = (as_of_date - pd.Timedelta(days=e)).date()
        cohort_start = (as_of_date - pd.Timedelta(days=e + (LOOKBACK_COHORTS - 1))).date()

        cohort_users = u_base.loc[
            (u_base['population'] == population) &
            (u_base['cost_date'] >= cohort_start) &
            (u_base['cost_date'] <= cohort_end)
        ][['population', 'cost_date', '__uid__']].copy()

        if bad_dates:
            cohort_users = cohort_users.loc[~cohort_users['cost_date'].isin(bad_dates)].copy()
        if excl:
            cohort_users = cohort_users.loc[~cohort_users['__uid__'].isin(excl)].copy()
        if cohort_users.empty:
            continue

        # CHANGED: unpack new return values (method, pct_used); day-steps still use config floor
        _, caps, _, _ = get_trimmed_cohort_and_caps(population, cohort_users, daily_user_cums, e)
        denom_w = (
            cohort_users.groupby(['population', 'cost_date'], observed=True)['__uid__']
                        .nunique().reset_index(name='N_users')
        )

        for k in range(start_k, e + 1):
            sum_prev = sum_cum_at_idx(
                daily_user_cums, cohort_users=cohort_users, idx=k - 2, caps=caps
            ).rename(columns={'sum_cum': 'sum_prev'})
            sum_curr = sum_cum_at_idx(
                daily_user_cums, cohort_users=cohort_users, idx=k - 1, caps=caps
            ).rename(columns={'sum_cum': 'sum_curr'})
            tmp = (
                denom_w
                .merge(sum_prev, on=['population', 'cost_date'])
                .merge(sum_curr, on=['population', 'cost_date'])
            )
            tmp['ARPU_prev']  = tmp['sum_prev'] / tmp['N_users']
            tmp['ARPU_curr']  = tmp['sum_curr'] / tmp['N_users']
            tmp['step_ratio'] = np.where(
                tmp['ARPU_prev'] > 0, tmp['ARPU_curr'] / tmp['ARPU_prev'], np.nan
            )
            mean_step, _, _ = weighted_mean_std_cv(
                tmp['step_ratio'].values, tmp['sum_prev'].values
            )
            step_rows.append({
                'population':      population,
                'day':             int(k),
                'growth_step':     float(mean_step),
                'effective_patch': f'{s}->{e}',
            })

    if not step_rows:
        return pd.DataFrame(cv_rows), pd.DataFrame()

    step_df = pd.DataFrame(step_rows).sort_values('day').reset_index(drop=True)

    # ── Base ARPU from first effective patch ──
    first      = effective[0]
    fs, fe     = first['s'], first['e']
    excl_first = first['excluded_snapshot']
    base_end   = (as_of_date - pd.Timedelta(days=fe)).date()
    base_start = (as_of_date - pd.Timedelta(days=fe + (LOOKBACK_COHORTS - 1))).date()

    base_users = u_base.loc[
        (u_base['population'] == population) &
        (u_base['cost_date'] >= base_start) &
        (u_base['cost_date'] <= base_end)
    ][['population', 'cost_date', '__uid__']].copy()
    bad_first = set(first['removed_dates'])
    if bad_first:
        base_users = base_users.loc[~base_users['cost_date'].isin(bad_first)].copy()
    if excl_first:
        base_users = base_users.loc[~base_users['__uid__'].isin(excl_first)].copy()

    # CHANGED: unpack new return values (method, pct_used); base ARPU still uses config floor
    _, base_caps, _, _ = get_trimmed_cohort_and_caps(population, base_users, daily_user_cums, fe)
    denom_base = (
        base_users.groupby(['population', 'cost_date'], observed=True)['__uid__']
                  .nunique().reset_index(name='N_users')
    )
    start_idx = 0 if fs == 1 else (fs - 1)
    sum_day1 = sum_cum_at_idx(
        daily_user_cums, cohort_users=base_users, idx=start_idx, caps=base_caps
    ).rename(columns={'sum_cum': 'sum_day1'})
    base = denom_base.merge(sum_day1, on=['population', 'cost_date'], how='inner')
    pooled_arpu_1 = (
        base['sum_day1'].sum() / base['N_users'].sum()
        if base['N_users'].sum() > 0 else 0.0
    )

    start_day = 1 if fs == 1 else fs
    out_rows  = [{'population': population, 'day': start_day,
                  'ARPU_nominal': float(pooled_arpu_1),
                  'growth_step':  np.nan,
                  'effective_patch': f'{fs}->{fe}'}]
    arpu = float(pooled_arpu_1)
    for _, row in step_df.iterrows():
        g = row['growth_step']
        if not np.isfinite(g):
            continue
        arpu *= float(g)
        out_rows.append({'population': population, 'day': int(row['day']),
                         'ARPU_nominal': float(arpu),
                         'growth_step':  float(g),
                         'effective_patch': row['effective_patch']})

    curve = pd.DataFrame(out_rows).sort_values('day').reset_index(drop=True)
    all_days = pd.DataFrame({'day': range(start_day, 366)})
    curve    = all_days.merge(curve, on='day', how='left')
    curve['population']      = population
    curve['effective_patch'] = curve['effective_patch'].ffill()
    curve['ARPU_nominal']    = curve['ARPU_nominal'].interpolate(
        method='linear', limit_area='inside'
    )
    curve = curve.dropna(subset=['ARPU_nominal']).reset_index(drop=True)
    curve['is_extrapolated'] = False

    return pd.DataFrame(cv_rows), curve


def extrapolate_curve_tail(curve, *, up_to_day=365, tail_days=30, debug=True):
    """Extend curve to up_to_day using geometric mean of last tail_days daily growth steps."""
    if curve.empty:
        return curve

    population    = curve['population'].iloc[0]
    last_real_day = int(curve['day'].max())
    if last_real_day >= up_to_day:
        return curve

    real = curve.loc[~curve['is_extrapolated']].sort_values('day')
    tail = real.tail(tail_days)
    if len(tail) < 2:
        if debug:
            print(f'  [{population}] Not enough tail data ({len(tail)}) to extrapolate.')
        return curve

    arpu_vals = tail['ARPU_nominal'].values
    ratios    = arpu_vals[1:] / arpu_vals[:-1]
    ratios    = ratios[np.isfinite(ratios) & (ratios > 0)]
    if len(ratios) == 0:
        return curve

    avg_daily_growth = float(np.exp(np.mean(np.log(ratios))))

    if debug:
        print(
            f'  [{population}] Extrapolating D{last_real_day + 1}→D{up_to_day}  '
            f'rate={avg_daily_growth:.6f}/day  (geom. mean of last {len(ratios)} daily ratios)'
        )

    last_arpu  = float(curve.loc[curve['day'] == last_real_day, 'ARPU_nominal'].iloc[0])
    last_patch = curve.loc[curve['day'] == last_real_day, 'effective_patch'].iloc[0]
    tag_patch  = f'{last_patch} (extrapolated)'

    new_rows = []
    arpu = last_arpu
    for d in range(last_real_day + 1, up_to_day + 1):
        arpu *= avg_daily_growth
        new_rows.append({
            'population':      population,
            'day':             d,
            'ARPU_nominal':    float(arpu),
            'growth_step':     avg_daily_growth,
            'effective_patch': tag_patch,
            'is_extrapolated': True,
        })

    return pd.concat([curve, pd.DataFrame(new_rows)], ignore_index=True)


def build_all_populations(
    u_base, daily_user_cums, *,
    as_of_date, debug=True,
    extrapolate_tail=False, tail_days=30, extrapolate_up_to=365,
):
    all_cv_rows = []
    all_curves  = []
    for pop in POPULATIONS:
        if debug:
            print(f'\n{"=" * 55}')
            print(f'POPULATION: {pop}')
            print(f'{"=" * 55}')
        cv_df, curve = build_curve(
            u_base, daily_user_cums,
            population=pop, as_of_date=as_of_date, debug=debug,
        )
        if not cv_df.empty:
            all_cv_rows.append(cv_df)
        if not curve.empty and extrapolate_tail:
            curve = extrapolate_curve_tail(
                curve, up_to_day=extrapolate_up_to, tail_days=tail_days, debug=debug,
            )
        if not curve.empty:
            all_curves.append(curve)
    cv_out    = pd.concat(all_cv_rows, ignore_index=True) if all_cv_rows else pd.DataFrame()
    curve_out = pd.concat(all_curves, ignore_index=True) if all_curves else pd.DataFrame()
    return cv_out, curve_out

print('Persistent trim curve builder defined.')


Persistent trim curve builder defined.


In [10]:
# ══════════════════════════════════════════════════════════════
# ORGANIC SHARE — multi-config cohort progression with scope
# Supports winsor, cohort_trim (with persistent trim), or none.
# Percentiles always computed from depositors only.
#
# If users_df has 'scope' and 'bucket' columns, uses them directly
# (RP: app vs non_app). Otherwise defaults to scope='all' and
# derives bucket from population == ORGANIC_LABEL (LS).
# ══════════════════════════════════════════════════════════════

def organic_share_cohort_progression(
    users_df, revenue_df, *,
    as_of_date,
    goal_horizons=GOAL_HORIZONS,
    checkpoints=CHECKPOINTS,
    lookback_cohorts=None,
    positive_amount_only=True,
    trim_configs,
    persistent_trim=True,
):
    if lookback_cohorts is None:
        lookback_cohorts = LOOKBACK_COHORTS
    as_of_date = pd.to_datetime(as_of_date).normalize()

    has_scope = 'scope' in users_df.columns and 'bucket' in users_df.columns
    cols = ['id', 'cost_date']
    if has_scope:
        cols += ['scope', 'bucket']
    if 'population' in users_df.columns:
        cols.append('population')

    u = users_df[cols].copy()
    u['cost_date'] = pd.to_datetime(u['cost_date'], errors='coerce').dt.date
    u = u.loc[pd.notna(u['cost_date'])].copy()

    if not has_scope:
        u['scope'] = 'all'
        u['bucket'] = np.where(u['population'] == ORGANIC_LABEL, 'organic', 'acquired')

    u = u.drop_duplicates(subset=['id'])

    r = revenue_df[['playerid', 'date', 'amount']].copy()
    r['date'] = pd.to_datetime(r['date'], errors='coerce').dt.date
    r = r.loc[pd.notna(r['date'])].copy()
    if positive_amount_only:
        r = r[r['amount'] > 0]

    rr = r.merge(
        u.rename(columns={'id': '__uid__'}),
        left_on='playerid', right_on='__uid__', how='inner'
    )
    rr['dsi'] = (pd.to_datetime(rr['date']) - pd.to_datetime(rr['cost_date'])).dt.days
    rr = rr.loc[(rr['dsi'] >= 0) & (rr['dsi'] <= (max(checkpoints) - 1))].copy()

    daily_user = (
        rr.groupby(['scope', 'bucket', 'cost_date', '__uid__', 'dsi'], observed=True)['amount']
          .sum().reset_index()
          .sort_values(['scope', 'bucket', 'cost_date', '__uid__', 'dsi'])
    )
    daily_user['cum_amount'] = (
        daily_user.groupby(['scope', 'bucket', 'cost_date', '__uid__'], observed=True)['amount']
                  .cumsum()
    )

    rows = []
    scopes = sorted(u['scope'].unique())

    for horizon in goal_horizons:
        cohort_end   = (as_of_date - pd.Timedelta(days=horizon)).date()
        cohort_start = (as_of_date - pd.Timedelta(days=horizon + (lookback_cohorts - 1))).date()

        elig = u.loc[
            (u['cost_date'] >= cohort_start) &
            (u['cost_date'] <= cohort_end)
        ].copy()

        if elig.empty:
            print(f'[horizon={horizon}] No eligible users — skipping.')
            continue

        eligible_cps = sorted(c for c in checkpoints if c <= horizon)

        for scope in scopes:
            scope_elig = elig.loc[elig['scope'] == scope]
            if scope_elig.empty:
                continue

            scope_du_h = daily_user.merge(
                scope_elig[['scope', 'bucket', 'id', 'cost_date']].rename(columns={'id': '__uid__'}),
                on=['scope', 'bucket', '__uid__', 'cost_date'], how='inner'
            )

            excluded = {
                cfg['label']: set()
                for cfg in trim_configs
                if cfg['method'] == 'cohort_trim' and persistent_trim
            }

            for cp in eligible_cps:
                du_cp = scope_du_h.loc[scope_du_h['dsi'] <= (cp - 1)]

                row = dict(
                    scope          = scope,
                    goal_horizon   = horizon,
                    cohort_start   = cohort_start,
                    cohort_end     = cohort_end,
                    checkpoint_day = cp,
                )

                if du_cp.empty:
                    for cfg in trim_configs:
                        lbl = cfg['label']
                        row.update({
                            f'organic_sum_{lbl}'       : 0.0,
                            f'acquired_sum_{lbl}'      : 0.0,
                            f'total_sum_{lbl}'         : 0.0,
                            f'organic_share_pct_{lbl}' : np.nan,
                            f'users_org_{lbl}'         : 0,
                            f'users_acq_{lbl}'         : 0,
                        })
                    rows.append(row)
                    continue

                cum_cp = (
                    du_cp.groupby(['bucket', 'cost_date', '__uid__'], observed=True)['cum_amount']
                         .max().reset_index(name='cum_cp')
                )
                cum_cp = (
                    scope_elig.rename(columns={'id': '__uid__'})
                              .merge(cum_cp, on=['bucket', 'cost_date', '__uid__'], how='left')
                )
                cum_cp['cum_cp'] = cum_cp['cum_cp'].fillna(0.0)

                print_line = f'  [{scope}] D{cp:>3}:'

                for cfg in trim_configs:
                    method, pct, lbl = cfg['method'], cfg['pct'], cfg['label']

                    if method == 'none' or pct == 0.0:
                        kept = cum_cp.copy()

                    elif method == 'cohort_trim':
                        working = cum_cp.copy()
                        if persistent_trim and excluded.get(lbl):
                            working = working.loc[~working['__uid__'].isin(excluded[lbl])]

                        depositors = working.loc[working['cum_cp'] > 0]
                        if not depositors.empty:
                            thresh_map = (
                                depositors.groupby('bucket', observed=True)['cum_cp']
                                          .quantile(1.0 - pct)
                                          .to_dict()
                            )
                            thresh_series = working['bucket'].map(thresh_map).fillna(np.inf)
                            above_mask = (working['cum_cp'] > 0) & (working['cum_cp'] > thresh_series)
                            if persistent_trim:
                                excluded[lbl] |= set(working.loc[above_mask, '__uid__'].unique())
                            kept = working.loc[~above_mask]
                        else:
                            kept = working

                    elif method == 'winsor':
                        depositors = cum_cp.loc[cum_cp['cum_cp'] > 0]
                        if not depositors.empty:
                            cap_map = (
                                depositors.groupby('bucket', observed=True)['cum_cp']
                                          .quantile(1.0 - pct)
                                          .to_dict()
                            )
                            kept = cum_cp.copy()
                            caps = kept['bucket'].map(cap_map).fillna(np.inf)
                            kept['cum_cp'] = np.minimum(kept['cum_cp'], caps)
                        else:
                            kept = cum_cp.copy()

                    else:
                        raise ValueError(f"Unknown trim method: {method}")

                    sums = kept.groupby('bucket', observed=True)['cum_cp'].sum().to_dict()
                    cnts = kept.groupby('bucket', observed=True)['__uid__'].nunique().to_dict()

                    org_sum = float(sums.get('organic',  0.0))
                    acq_sum = float(sums.get('acquired', 0.0))
                    total   = org_sum + acq_sum
                    share   = (org_sum / total) if total > 0 else np.nan

                    row.update({
                        f'organic_sum_{lbl}'       : org_sum,
                        f'acquired_sum_{lbl}'      : acq_sum,
                        f'total_sum_{lbl}'         : total,
                        f'organic_share_pct_{lbl}' : share,
                        f'users_org_{lbl}'         : int(cnts.get('organic',  0)),
                        f'users_acq_{lbl}'         : int(cnts.get('acquired', 0)),
                    })

                    print_line += (f'  [{lbl}] org={org_sum:>10,.0f} '
                                   f'acq={acq_sum:>10,.0f} share={share:.1%}')

                print(print_line)
                rows.append(row)

    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows).sort_values(['scope', 'goal_horizon', 'checkpoint_day']).reset_index(drop=True)
    id_cols = ['scope', 'goal_horizon', 'cohort_start', 'cohort_end', 'checkpoint_day']
    metric_cols = []
    for cfg in trim_configs:
        lbl = cfg['label']
        metric_cols += [
            f'organic_sum_{lbl}', f'acquired_sum_{lbl}', f'total_sum_{lbl}',
            f'organic_share_pct_{lbl}', f'users_org_{lbl}', f'users_acq_{lbl}',
        ]
    return df[id_cols + metric_cols]


print('Organic share function defined.')


Organic share function defined.


In [11]:
# ══════════════════════════════════════════════════════════════
# PART 4 helpers — Goal construction + brand pipeline
# Per-population: adjusted = (ARPU_day / ARPU_horizon) × (1 − organic_share_at_horizon)
# Blended:        adjusted = raw_goal_ratio (no organic adjustment)
# ──
# Organic share rule: for every day inside a horizon, use the share measured AT
# the horizon endpoint (not the share at the day's nearest checkpoint).
# RP: pin lookup to min(horizon, ORGANIC_SHARE_CAP_HORIZON) when cap is set.
# LS: no cap (cap is None).
# ══════════════════════════════════════════════════════════════

def build_goals(curve_df, organic_df, populations, goal_horizons=None,
                organic_share_cap_horizon=None):
    if goal_horizons is None:
        goal_horizons = GOAL_HORIZONS

    has_scope = 'scope' in organic_df.columns
    available_scopes = set(organic_df['scope'].unique()) if has_scope else {'all'}

    org_lookup = {}
    scope_col = 'scope' if has_scope else None
    endpoint = organic_df.loc[organic_df['checkpoint_day'] == organic_df['goal_horizon']]
    if scope_col:
        for scope, grp in endpoint.groupby(scope_col):
            org_lookup[scope] = grp.set_index('goal_horizon')['organic_share_pct'].to_dict()
    else:
        org_lookup['all'] = endpoint.set_index('goal_horizon')['organic_share_pct'].to_dict()

    def pop_to_scope(pop):
        if pop == 'App' and 'app' in available_scopes:
            return 'app'
        if 'non_app' in available_scopes:
            return 'non_app'
        return next(iter(available_scopes))

    rows = []
    all_pops = list(populations) + ['Blended']

    for pop in all_pops:
        is_blended = (pop == 'Blended')

        pop_curve = curve_df.loc[curve_df['population'] == pop].copy()
        if pop_curve.empty:
            continue
        pop_curve = pop_curve.drop_duplicates(subset='day').set_index('day')

        for horizon in goal_horizons:
            if horizon not in pop_curve.index:
                print(f'[{pop}] Day {horizon} missing from curve — skipping horizon.')
                continue
            arpu_horizon = pop_curve.loc[horizon, 'ARPU_nominal']
            if not np.isfinite(arpu_horizon) or arpu_horizon == 0:
                continue

            if is_blended:
                org_share = 0.0
            else:
                scope = pop_to_scope(pop)
                if organic_share_cap_horizon is not None:
                    lookup_horizon = min(horizon, organic_share_cap_horizon)
                else:
                    lookup_horizon = horizon
                org_share = org_lookup.get(scope, {}).get(lookup_horizon, np.nan)

            for day in range(1, horizon + 1):
                if day not in pop_curve.index:
                    continue
                arpu_day  = pop_curve.loc[day, 'ARPU_nominal']
                eff_patch = pop_curve.loc[day, 'effective_patch'] if 'effective_patch' in pop_curve.columns else ''
                is_extrap = bool(pop_curve.loc[day, 'is_extrapolated']) if 'is_extrapolated' in pop_curve.columns else False
                raw_goal  = arpu_day / arpu_horizon

                if np.isfinite(org_share):
                    adj_goal = raw_goal * (1 - org_share)
                else:
                    adj_goal = np.nan

                rows.append(dict(
                    population           = pop,
                    goal_horizon         = horizon,
                    day                  = day,
                    ARPU_nominal         = float(arpu_day),
                    ARPU_at_horizon      = float(arpu_horizon),
                    raw_goal_ratio       = float(raw_goal),
                    organic_share        = float(org_share) if np.isfinite(org_share) else None,
                    adjusted_goal_ratio  = float(adj_goal)  if np.isfinite(adj_goal)  else None,
                    effective_patch      = eff_patch,
                    is_extrapolated      = is_extrap,
                ))

    return pd.DataFrame(rows)


def _banner(title: str):
    print('\n' + '─' * 60)
    print(title)
    print('─' * 60)


def monitor_show_users(brand, users_df):
    """After SQL load: population counts + date span."""
    _banner(f'[{brand}] MONITOR · DATA LOAD')
    print(f'users_df rows:   {len(users_df):,}')
    print('population counts:')
    print(users_df['population'].value_counts().to_string())
    cd = pd.to_datetime(users_df['cost_date'], errors='coerce')
    print(f"cost_date span: {cd.min().date()} → {cd.max().date()}")
    if 'scope' in users_df.columns:
        print('\nscope × bucket:')
        print(users_df.groupby(['scope', 'bucket'], observed=True).size().to_string())


def monitor_show_cv_curve(brand, stage, cv_df, curve_df, populations):
    """After ARPU curve build: CV table + milestone ARPU."""
    _banner(f'[{brand}] MONITOR · {stage}')
    if cv_df is not None and not cv_df.empty:
        display_cols = [
            'population', 'patch', 'n_dates_before', 'n_dates_after',
            'cv_before', 'cv_after', 'removed_dates',
            'total_rev_before_trim', 'total_rev_after_trim',
            'flagged', 'trim_method',
            # ADDED: winsor escalation diagnostics
            'floor_pct', 'pct_used', 'escalated',
            'capped_by_revenue_limit', 'revenue_cut_fraction',
        ]
        display_cols = [c for c in display_cols if c in cv_df.columns]
        print('CV summary:')
        print(cv_df[display_cols].to_string(index=False))
        if 'flagged' in cv_df.columns:
            flagged = cv_df.loc[cv_df['flagged'] == True]
            if not flagged.empty:
                print('\n>>> FLAGGED PATCHES:')
                for _, r in flagged.iterrows():
                    patch = r['patch'] if 'patch' in r.index else '?'
                    print(f"    {r.get('population', '?')} / {patch}  CV={r.get('cv_after', float('nan')):.4f}")
            else:
                print('\nNo flagged patches in this stage.')
    else:
        print('CV summary: (empty)')

    if curve_df is not None and not curve_df.empty:
        milestones = list(MONITOR_PREVIEW_DAYS)
        for pop in populations:
            sub = curve_df.loc[curve_df['population'] == pop]
            if sub.empty:
                continue
            cols = [c for c in ['day', 'ARPU_nominal', 'effective_patch', 'is_extrapolated'] if c in sub.columns]
            ms = sub.loc[sub['day'].isin(milestones)][cols]
            print(f'\nARPU curve milestones — {pop}  (n_days={sub["day"].nunique()})')
            print(ms.to_string(index=False))
            if 'is_extrapolated' in sub.columns and sub['is_extrapolated'].any():
                n_ex = int(sub['is_extrapolated'].sum())
                last_real = int(sub.loc[~sub['is_extrapolated'], 'day'].max()) if (~sub['is_extrapolated']).any() else None
                print(f'  extrapolated days: {n_ex:,}  last real day: {last_real}')
    else:
        print('ARPU curve: (empty)')


def monitor_show_organic(brand, organic_df):
    """After organic share computation."""
    _banner(f'[{brand}] MONITOR · ORGANIC SHARE')
    if organic_df is None or organic_df.empty:
        print('organic_df: (empty)')
        return
    print(f'rows: {len(organic_df):,}')
    # endpoint only (checkpoint_day == goal_horizon) — what goals actually use
    if 'checkpoint_day' in organic_df.columns and 'goal_horizon' in organic_df.columns:
        ep = organic_df.loc[organic_df['checkpoint_day'] == organic_df['goal_horizon']].copy()
        cols = [c for c in ['scope', 'goal_horizon', 'organic_share_pct', 'users_org', 'users_acq'] if c in ep.columns]
        for scope in ep['scope'].unique() if 'scope' in ep.columns else [None]:
            sub = ep if scope is None else ep.loc[ep['scope'] == scope]
            title = f'endpoint shares — scope={scope}' if scope is not None else 'endpoint shares'
            print(f'\n{title}:')
            print(sub[cols].to_string(index=False))
    else:
        print(organic_df.head(20).to_string(index=False))


def monitor_show_goals(brand, goals_df, populations):
    """After goal construction: sample horizons × days."""
    _banner(f'[{brand}] MONITOR · GOALS')
    if goals_df is None or goals_df.empty:
        print('goals_df: (empty)')
        return
    print(f'total goal rows: {len(goals_df):,}')
    print('counts by population:')
    print(goals_df.groupby('population', observed=True).size().to_string())

    preview_days = list(MONITOR_PREVIEW_DAYS)
    preview_horizons = list(MONITOR_PREVIEW_HORIZONS)
    preview_cols = [
        c for c in [
            'population', 'goal_horizon', 'day',
            'raw_goal_ratio', 'organic_share', 'adjusted_goal_ratio',
            'ARPU_nominal', 'is_extrapolated',
        ] if c in goals_df.columns
    ]
    for pop in list(populations) + ['Blended']:
        for horizon in preview_horizons:
            sub = goals_df.loc[
                (goals_df['population'] == pop) &
                (goals_df['goal_horizon'] == horizon) &
                (goals_df['day'].isin(preview_days))
            ]
            if sub.empty:
                continue
            print(f'\n── {pop} │ horizon D{horizon} ──')
            print(sub[preview_cols].to_string(index=False))


def run_brand_pipeline(cfg, users_df, revenue_df, as_of_date=AS_OF_DATE, monitor=None):
    """Parts 1-4 for one brand. Returns dict of frames with brand column.

    monitor=True  -> print intermediate tables after every part (for Colab monitoring)
    monitor=False -> banners only
    monitor=None  -> use global MONITOR_STEPS
    """
    global POPULATIONS  # reassigned briefly for Blended pass; must be declared before first use

    if monitor is None:
        monitor = bool(globals().get('MONITOR_STEPS', True))

    apply_brand_globals(cfg)
    brand = cfg['brand']

    if monitor:
        monitor_show_users(brand, users_df)

    # ── PART 1 — per-population ARPU curves ──
    _banner(f'[{brand}] PART 1/4 — per-population ARPU curves')
    print(f'[{brand}] Building base cums + adaptive CV patches (debug lines below)...')
    u_pop, daily_pop = build_user_revenue_cums(
        users_df.loc[users_df['population'].isin(POPULATIONS)].copy(),
        revenue_df,
        max_day=365,
    )
    print(f'[{brand}] base users in ARPU pops: {len(u_pop):,}  daily_user rows: {len(daily_pop):,}')

    cv_pop_df, curve_pop_df = build_all_populations(
        u_pop, daily_pop,
        as_of_date=as_of_date,
        extrapolate_tail=EXTRAPOLATE_TAIL,
        tail_days=EXTRAPOLATION_TAIL_DAYS,
        extrapolate_up_to=365,
        debug=True,
    )
    if monitor:
        monitor_show_cv_curve(brand, 'PART 1 done — per-pop CV + curve', cv_pop_df, curve_pop_df, POPULATIONS)

    # ── PART 2 — Blended ──
    _banner(f'[{brand}] PART 2/4 — Blended ARPU curve')
    print(f'[{brand}] Building BLENDED...')
    users_blend = users_df.copy()
    users_blend['population'] = 'Blended'
    u_blend, daily_blend = build_user_revenue_cums(
        users_blend, revenue_df, max_day=365
    )
    print(f'[{brand}] blended users: {len(u_blend):,}')
    # Temporarily set POPULATIONS so build_all_populations iterates Blended only
    _BLENDED_POPS_BACKUP = list(POPULATIONS)
    POPULATIONS = ['Blended']
    try:
        cv_blend_df, curve_blend_df = build_all_populations(
            u_blend, daily_blend,
            as_of_date=as_of_date,
            extrapolate_tail=EXTRAPOLATE_TAIL,
            tail_days=EXTRAPOLATION_TAIL_DAYS,
            extrapolate_up_to=365,
            debug=True,
        )
    finally:
        POPULATIONS = _BLENDED_POPS_BACKUP

    if monitor:
        monitor_show_cv_curve(brand, 'PART 2 done — Blended CV + curve', cv_blend_df, curve_blend_df, ['Blended'])

    cv_df    = pd.concat([cv_pop_df,    cv_blend_df],    ignore_index=True)
    curve_df = pd.concat([curve_pop_df, curve_blend_df], ignore_index=True)

    # ── PART 3 — organic share ──
    _banner(f'[{brand}] PART 3/4 — organic share')
    org_label = f'{ORGANIC_TRIM_METHOD}_{round(ORGANIC_TRIM_PCT * 100):g}pct'
    org_config = [{'method': ORGANIC_TRIM_METHOD, 'pct': ORGANIC_TRIM_PCT, 'label': org_label}]
    print(f'[{brand}] Computing organic share [{org_label}] for horizons {GOAL_HORIZONS}...')
    organic_full = organic_share_cohort_progression(
        users_df, revenue_df,
        as_of_date=as_of_date,
        goal_horizons=GOAL_HORIZONS,
        checkpoints=CHECKPOINTS,
        lookback_cohorts=LOOKBACK_COHORTS,
        positive_amount_only=True,
        trim_configs=org_config,
        persistent_trim=True,
    )
    organic_df = organic_full[['scope', 'goal_horizon', 'cohort_start', 'cohort_end',
                               'checkpoint_day',
                               f'organic_share_pct_{org_label}',
                               f'users_org_{org_label}',
                               f'users_acq_{org_label}']].copy()
    organic_df = organic_df.rename(columns={
        f'organic_share_pct_{org_label}': 'organic_share_pct',
        f'users_org_{org_label}': 'users_org',
        f'users_acq_{org_label}': 'users_acq',
    })
    if monitor:
        monitor_show_organic(brand, organic_df)
    else:
        print(f'[{brand}] organic rows: {len(organic_df):,}')

    # ── PART 4 — goals ──
    _banner(f'[{brand}] PART 4/4 — goal construction')
    goals_df = build_goals(
        curve_df, organic_df, POPULATIONS,
        goal_horizons=GOAL_HORIZONS,
        organic_share_cap_horizon=ORGANIC_SHARE_CAP_HORIZON,
    )
    if monitor:
        monitor_show_goals(brand, goals_df, POPULATIONS)
    else:
        print(f'[{brand}] goals rows: {len(goals_df):,}')

    # stamp brand
    for df in (cv_df, curve_df, organic_df, goals_df):
        if not df.empty:
            df.insert(0, 'brand', brand)

    print(f'\n[{brand}] COMPLETE — goals {len(goals_df):,} | curve {len(curve_df):,} | organic {len(organic_df):,} | cv {len(cv_df):,}')
    return {
        'cv': cv_df,
        'curve': curve_df,
        'organic': organic_df,
        'goals': goals_df,
        'populations': list(cfg['populations']),
    }

print('run_brand_pipeline + monitors ready.')
print('Set MONITOR_STEPS=True in config cell to print intermediate tables after each part.')


run_brand_pipeline + monitors ready.
Set MONITOR_STEPS=True in config cell to print intermediate tables after each part.


In [12]:
# ══════════════════════════════════════════════════════════════
# LS APP + WINSOR WIRING (this Colab only)
# ══════════════════════════════════════════════════════════════
# 1) Wire pct_used from the CV pass into curve day-steps (v2 archive
#    still builds the curve on the config-floor winsor).
# 2) LS App: keep winsor at 0% — do not climb the escalation ladder.
# 3) native_early_rp_tail: keep LS App native through last measured day,
#    then RP App day-to-day growth.

_orig_patch_cv_adaptive = patch_cv_adaptive
_orig_get_trimmed = get_trimmed_cohort_and_caps
_pct_used_by_pop_e = {}


def patch_cv_adaptive(*args, **kwargs):
    global WINSOR_ESCALATION_STEPS
    population = kwargs.get('population')
    saved_steps = list(WINSOR_ESCALATION_STEPS)
    if ACTIVE_BRAND == 'lonestar' and population == 'App':
        WINSOR_ESCALATION_STEPS = [0.0]
    try:
        patch, stats, removed, flagged, newly = _orig_patch_cv_adaptive(*args, **kwargs)
    finally:
        WINSOR_ESCALATION_STEPS = saved_steps
    if stats:
        pop = stats.get('population') or population
        e = kwargs.get('e')
        if e is None and stats.get('patch'):
            try:
                e = int(str(stats['patch']).split('->')[1])
            except Exception:
                e = None
        if pop is not None and e is not None and 'pct_used' in stats:
            _pct_used_by_pop_e[(pop, int(e))] = stats['pct_used']
    return patch, stats, removed, flagged, newly


def get_trimmed_cohort_and_caps(population, cohort_users, daily_user_cums, e, pct_override=None):
    if pct_override is None:
        pct_override = _pct_used_by_pop_e.get((population, int(e)))
    return _orig_get_trimmed(
        population, cohort_users, daily_user_cums, e, pct_override=pct_override
    )


def _curve_for_pop(curve_df, population):
    out = curve_df.loc[curve_df['population'] == population].copy()
    if out.empty:
        return out
    return out.drop_duplicates(subset='day').sort_values('day').reset_index(drop=True)


def last_measured_day(native):
    """Last life day from measured patches, not LS tail extrapolation."""
    if native.empty:
        raise RuntimeError('Native LS App curve is empty.')
    if 'is_extrapolated' in native.columns:
        flag = native['is_extrapolated'].fillna(False).astype(bool)
        real = native.loc[~flag]
        if not real.empty:
            return int(real['day'].max())
    return int(native['day'].max())


def native_early_rp_tail_curve(native_curve, rp_app_curve, *, target_pop='App', max_day=None):
    """Keep native LS App through last measured day; dress RP App growth after that.

    After splice day S:
      ARPU(d) = ARPU(d-1) × (ARPU_RP(d) / ARPU_RP(d-1))
    """
    native = _curve_for_pop(native_curve, target_pop)
    rp = _curve_for_pop(rp_app_curve, 'App')
    if native.empty:
        raise RuntimeError('native_early_rp_tail needs a native LS App curve.')
    if rp.empty:
        raise RuntimeError('native_early_rp_tail needs an RP App curve.')

    horizon = int(max_day) if max_day is not None else max(GOAL_HORIZONS)
    splice = last_measured_day(native)
    native = native.sort_values('day')
    rp = rp.drop_duplicates(subset='day').set_index('day')

    keep = native.loc[native['day'] <= splice].copy()
    keep['population'] = target_pop
    keep['is_extrapolated'] = False

    last_arpu = float(keep.loc[keep['day'] == splice, 'ARPU_nominal'].iloc[0])
    extra_rows = []
    for day in range(splice + 1, horizon + 1):
        if day not in rp.index or (day - 1) not in rp.index:
            continue
        rp_prev = float(rp.loc[day - 1, 'ARPU_nominal'])
        rp_now = float(rp.loc[day, 'ARPU_nominal'])
        growth = 1.0 if (not np.isfinite(rp_prev) or rp_prev <= 0) else rp_now / rp_prev
        last_arpu = last_arpu * growth
        row = rp.loc[day].to_dict()
        row['day'] = day
        row['population'] = target_pop
        row['ARPU_nominal'] = last_arpu
        row['is_extrapolated'] = True
        extra_rows.append(row)

    out = pd.concat([keep, pd.DataFrame(extra_rows)], ignore_index=True)
    out = out.sort_values('day').drop_duplicates(subset='day').reset_index(drop=True)
    out['brand'] = 'lonestar'
    out['population'] = target_pop
    end_day = int(out['day'].max())
    print(
        f'native_early_rp_tail: splice after day {splice}  '
        f'ARPU({splice})=${float(keep.loc[keep["day"] == splice, "ARPU_nominal"].iloc[0]):.4f}  '
        f'ARPU({end_day})=${float(out.loc[out["day"] == end_day, "ARPU_nominal"].iloc[0]):.2f}'
    )
    return out, splice

print('LS App helpers ready: pct_used wired, App winsor locked at 0%, native_early_rp_tail defined.')


LS App helpers ready: pct_used wired, App winsor locked at 0%, native_early_rp_tail defined.


In [13]:
# ══════════════════════════════════════════════════════════════
# RUN — RP + LS winsor_esc, then LS App native_early_rp_tail
# ══════════════════════════════════════════════════════════════
import copy

brand_results = {}
_pct_used_by_pop_e.clear()

print('\n[realprize] capped winsor_esc (all pops)')
rp_cfg = copy.deepcopy(BRAND_CONFIGS['realprize'])
rp_users, rp_rev = load_brand_tables(rp_cfg, as_of_date=AS_OF_DATE)
rp_res = run_brand_pipeline(rp_cfg, rp_users, rp_rev, as_of_date=AS_OF_DATE)
brand_results['realprize'] = rp_res

print('\n[lonestar] capped winsor_esc Web + Affiliate + Blended (no App)')
_pct_used_by_pop_e.clear()
ls_cfg = copy.deepcopy(BRAND_CONFIGS['lonestar'])
ls_users, ls_rev = load_brand_tables(ls_cfg, as_of_date=AS_OF_DATE)
ls_core_cfg = copy.deepcopy(ls_cfg)
ls_core_cfg['populations'] = ['Web', 'Affiliate']
ls_core = run_brand_pipeline(ls_core_cfg, ls_users, ls_rev, as_of_date=AS_OF_DATE)
brand_results['lonestar'] = ls_core

print('\n[lonestar/App] native measured patches (winsor 0%), then RP App tail')
_pct_used_by_pop_e.clear()
ls_app_cfg = copy.deepcopy(ls_cfg)
ls_app_cfg['populations'] = ['App']
# Temporary: LS App post-floor history is shorter than the LS 20-date gate.
# Web / Affiliate / Blended keep min_cohort_dates = 20.
ls_app_cfg['min_cohort_dates'] = 1
print(f"[lonestar/App] start_date={LS_APP_START_DATE.date()}  "
      f"min_cohort_dates={ls_app_cfg['min_cohort_dates']} "
      f"(Web/Aff stay {ls_cfg['min_cohort_dates']})")
ls_app_native = run_brand_pipeline(ls_app_cfg, ls_users, ls_rev, as_of_date=AS_OF_DATE)

spliced, splice_day = native_early_rp_tail_curve(
    ls_app_native['curve'],
    rp_res['curve'],
    max_day=max(GOAL_HORIZONS),
)
apply_brand_globals(ls_cfg)
app_goals = build_goals(
    spliced,
    ls_core['organic'],
    ['App'],
    goal_horizons=GOAL_HORIZONS,
    organic_share_cap_horizon=None,
)
if 'brand' not in app_goals.columns:
    app_goals.insert(0, 'brand', 'lonestar')
else:
    app_goals['brand'] = 'lonestar'
# Provisional: no App organic haircut until App has enough horizon history.
app_goals['organic_share'] = 0.0
app_goals['adjusted_goal_ratio'] = app_goals['raw_goal_ratio']
print(f'[lonestar/App] splice_day={splice_day}  goals={len(app_goals):,}  organic=0 (off)')

# App pipeline always builds Blended in PART 2. Keep App CV only
# so lonestar/Blended is not duplicated in the export.
ls_app_cv = ls_app_native['cv']
if not ls_app_cv.empty and 'population' in ls_app_cv.columns:
    ls_app_cv = ls_app_cv.loc[ls_app_cv['population'] == 'App'].copy()
cv_df = pd.concat(
    [rp_res['cv'], ls_core['cv'], ls_app_cv],
    ignore_index=True,
)
curve_df = pd.concat(
    [rp_res['curve'], ls_core['curve'], spliced],
    ignore_index=True,
)
organic_df = pd.concat([rp_res['organic'], ls_core['organic']], ignore_index=True)
goals_detail_df = pd.concat(
    [rp_res['goals'], ls_core['goals'], app_goals],
    ignore_index=True,
)

MAIN_GOAL_COLS = [
    'brand', 'population', 'goal_horizon', 'day',
    'raw_goal_ratio', 'organic_share', 'adjusted_goal_ratio',
]
goals_df = (
    goals_detail_df[MAIN_GOAL_COLS].copy()
    if not goals_detail_df.empty
    else pd.DataFrame(columns=MAIN_GOAL_COLS)
)

print('\n' + '=' * 60)
print('UNIFIED RUN COMPLETE — v2 winsor_esc + LS App native_early_rp_tail')
print(f'  as_of_date     = {AS_OF_DATE.date()}')
print(f'  brands         = {RUN_BRANDS}')
print(f'  LS App splice  = day {splice_day}')
print(f'  LS App start   = {LS_APP_START_DATE.date()}')
print(f'  LS App min_cohort_dates = {ls_app_cfg["min_cohort_dates"]} (temporary)')
print(f'  cv rows        = {len(cv_df):,}')
print(f'  curve rows     = {len(curve_df):,}')
print(f'  organic rows   = {len(organic_df):,}')
print(f'  goals (main)   = {len(goals_df):,}')
print(f'  goals (detail) = {len(goals_detail_df):,}')
print('=' * 60)

if not goals_df.empty:
    print('\nRow counts by brand × population:')
    print(
        goals_df.groupby(['brand', 'population'], observed=True)
                .size().rename('n_rows').reset_index()
                .to_string(index=False)
    )



[realprize] capped winsor_esc (all pops)
[realprize] SQL floor date: 2025-07-10  (AS_OF_DATE = 2026-08-19)
Downloading: 100%|██████████|
Downloading: 100%|██████████|
[realprize] users_df:   928,418 rows
[realprize] revenue_df: 1,766,095 rows
population
Affiliate    289313
App          257393
Organic      190207
Web          118555
PPC           72950
[realprize] experiment: channel_type=app_affiliate → Affiliate; other APP stays App. LoneStar unchanged.
population  scope    bucket  
Affiliate   non_app  acquired    289313
App         app      acquired    212937
                     organic      44456
Organic     non_app  organic     190207
PPC         non_app  acquired     72950
Web         non_app  acquired    118555

BRAND: realprize
  populations = ['Web', 'App', 'Affiliate']
  CV threshold = 0.15  good_enough = 0.1
  min_cohort_dates = 1
  extrapolate_tail = False
  organic_share_cap_horizon = 120

────────────────────────────────────────────────────────────
[realprize] MONITOR ·

In [14]:
# ══════════════════════════════════════════════════════════════
# PREVIEW — main goals table + Combined-style spot checks
# ══════════════════════════════════════════════════════════════

print('MAIN GOALS (head):')
print(goals_df.head(20).to_string(index=False))

preview_days = [1, 7, 14, 30, 60, 90, 120, 150, 180, 210, 240, 270, 365]
preview_horizons = [7, 30, 120, 365]

for brand in RUN_BRANDS:
    pops = list(BRAND_CONFIGS[brand]['populations']) + ['Blended']
    if brand == 'lonestar' and 'App' not in pops:
        pops = pops[:-1] + ['App', 'Blended']
    for pop in pops:
        for horizon in preview_horizons:
            sub = goals_df.loc[
                (goals_df['brand'] == brand) &
                (goals_df['population'] == pop) &
                (goals_df['goal_horizon'] == horizon) &
                (goals_df['day'].isin(preview_days))
            ]
            if sub.empty:
                continue
            print(f'\n── {brand} │ {pop} │ horizon D{horizon} ──')
            print(sub[MAIN_GOAL_COLS].to_string(index=False))

# CV flags
if not cv_df.empty and 'flagged' in cv_df.columns:
    flagged = cv_df.loc[cv_df['flagged'] == True]
    if not flagged.empty:
        print('\n>>> FLAGGED PATCHES:')
        cols = [c for c in ['brand', 'population', 'patch', 'cv_after'] if c in flagged.columns]
        print(flagged[cols].to_string(index=False))
    else:
        print('\nNo flagged patches.')

# ADDED: winsor escalation report (one row per brand × population × patch)
if not cv_df.empty:
    esc_cols = [
        c for c in [
            'brand', 'population', 'patch',
            'floor_pct', 'pct_used', 'escalated',
            'capped_by_revenue_limit', 'revenue_cut_fraction',
            'cv_before', 'cv_after', 'flagged',
        ] if c in cv_df.columns
    ]
    print('\n>>> WINSOR ESCALATION REPORT:')
    print(cv_df[esc_cols].to_string(index=False))


MAIN GOALS (head):
    brand population  goal_horizon  day  raw_goal_ratio  organic_share  adjusted_goal_ratio
realprize        Web             7    1        0.518819       0.515831             0.251196
realprize        Web             7    2        0.698254       0.515831             0.338073
realprize        Web             7    3        0.783905       0.515831             0.379542
realprize        Web             7    4        0.859652       0.515831             0.416217
realprize        Web             7    5        0.912072       0.515831             0.441597
realprize        Web             7    6        0.955966       0.515831             0.462849
realprize        Web             7    7        1.000000       0.515831             0.484169
realprize        Web            30    1        0.318435       0.500838             0.158951
realprize        Web            30    2        0.428567       0.500838             0.213925
realprize        Web            30    3        0.481137      

In [15]:
# ══════════════════════════════════════════════════════════════
# EXPORT — unique run folder when re-run same day
# ══════════════════════════════════════════════════════════════
# Naming when you run more than once per day:
#   runs/<as_of>_<brands>_<run_ts>/
#   e.g. 2026-08-05_rp_142233/   or  2026-08-05_rp_ls_143015/
# run_ts = local clock when this export cell runs (HHMMSS) — never overwrites.
# as_of_date is the cohort anchor (today-2), not wall-clock time.
# ══════════════════════════════════════════════════════════════

from datetime import datetime
from pathlib import Path
import os
import re

# Brand slug: realprize → rp, lonestar → ls (stable short codes)
_BRAND_SLUG = {'realprize': 'rp', 'lonestar': 'ls'}
brand_slug = '_'.join(
    _BRAND_SLUG.get(b, re.sub(r'[^a-z0-9]+', '', b.lower())[:6])
    for b in RUN_BRANDS
)

# Unique per execution
exported_at = datetime.now()
run_ts = exported_at.strftime('%H%M%S')
run_tag = f"{AS_OF_DATE.date()}_{brand_slug}_rp_app_affiliate_to_aff_{run_ts}"

main_goals_path    = f'combined_goals_{run_tag}.csv'
detail_goals_path  = f'combined_goals_detail_{run_tag}.csv'
curve_path         = f'combined_arpu_curve_{run_tag}.csv'
organic_path       = f'combined_organic_share_{run_tag}.csv'
# CHANGED: distinct filename so this adaptive-winsor test does not overwrite baseline cv_summary
cv_path            = f'combined_cv_summary_{run_tag}_adaptive_test.csv'

goals_df.to_csv(main_goals_path, index=False)
goals_detail_df.to_csv(detail_goals_path, index=False)
curve_df.to_csv(curve_path, index=False)
organic_df.to_csv(organic_path, index=False)
cv_df.to_csv(cv_path, index=False)

print(f'Run tag: {run_tag}')
print(f'  as_of_date (data) = {AS_OF_DATE.date()}')
print(f'  brands            = {RUN_BRANDS}')
print(f'  run_ts (export)   = {run_ts}  (unique each time this cell runs)')
print('Saved in working directory:')
for pth in [main_goals_path, detail_goals_path, curve_path, organic_path, cv_path]:
    print(f'  {pth}')

run_candidates = [
    Path('/content/drive/MyDrive/lee_project/marketing_goals/runs'),
    Path('/content/drive/My Drive/lee_project/marketing_goals/runs'),
]
drive_runs = next((c for c in run_candidates if c.is_dir()), None)

if drive_runs is not None:
    out_dir = drive_runs / run_tag
    out_dir.mkdir(parents=True, exist_ok=True)
    goals_df.to_csv(out_dir / 'combined_goals.csv', index=False)
    goals_detail_df.to_csv(out_dir / 'combined_goals_detail.csv', index=False)
    curve_df.to_csv(out_dir / 'combined_arpu_curve.csv', index=False)
    organic_df.to_csv(out_dir / 'combined_organic_share.csv', index=False)
    # CHANGED: adaptive-test name — keep side-by-side with prior combined_cv_summary.csv
    cv_df.to_csv(out_dir / 'combined_cv_summary_adaptive_test.csv', index=False)
    meta = pd.DataFrame([{
        'run_tag': run_tag,
        'as_of_date': str(AS_OF_DATE.date()),
        'brands': ','.join(RUN_BRANDS),
        'brand_slug': brand_slug,
        'run_ts': run_ts,
        'exported_at': exported_at.isoformat(timespec='seconds'),
        'main_columns': ','.join(MAIN_GOAL_COLS),
        'env': 'colab',
        'engine': 'v2 winsor_esc + lonestar/App native_early_rp_tail',
        'ls_app_winsor': '0% locked (no escalation)',
        'ls_app_organic': 'off (organic_share=0, adjusted=raw)',
        'ls_blended_includes_app': False,
        'ls_app_splice_day': int(splice_day) if 'splice_day' in dir() else None,
        'ls_app_start_date': str(LS_APP_START_DATE.date()),
        'ls_app_min_cohort_dates': 1,
        'ls_core_min_cohort_dates': 20,
        'ls_app_pre_floor_affid1': 'Affiliate',
    }])
    meta.to_csv(out_dir / 'run_meta.csv', index=False)
    print(f'\nAlso wrote to: {out_dir}')
    print('(Each re-run creates a NEW folder under runs/ — previous runs stay.)')
else:
    print('\nDrive runs/ not found (mount skip or wrong path).')
    print('Browser downloads below still give you the full pack.')

from google.colab import files
for pth in [main_goals_path, detail_goals_path, curve_path, organic_path, cv_path]:
    files.download(pth)
print('Downloads started in browser.')


Run tag: 2026-08-19_rp_ls_rp_app_affiliate_to_aff_112133
  as_of_date (data) = 2026-08-19
  brands            = ['realprize', 'lonestar']
  run_ts (export)   = 112133  (unique each time this cell runs)
Saved in working directory:
  combined_goals_2026-08-19_rp_ls_rp_app_affiliate_to_aff_112133.csv
  combined_goals_detail_2026-08-19_rp_ls_rp_app_affiliate_to_aff_112133.csv
  combined_arpu_curve_2026-08-19_rp_ls_rp_app_affiliate_to_aff_112133.csv
  combined_organic_share_2026-08-19_rp_ls_rp_app_affiliate_to_aff_112133.csv
  combined_cv_summary_2026-08-19_rp_ls_rp_app_affiliate_to_aff_112133_adaptive_test.csv

Also wrote to: /content/drive/MyDrive/lee_project/marketing_goals/runs/2026-08-19_rp_ls_rp_app_affiliate_to_aff_112133
(Each re-run creates a NEW folder under runs/ — previous runs stay.)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloads started in browser.
